# EuroSAT SAR ↔ Optical Hybrid Retrieval Training

This notebook follows the same structure and code format as the Optical ↔ Multispectral notebook, but trains:

- 2-band Sentinel-1 SAR (VV, VH)
- Optical RGB

It saves:

- SAR and optical encoders
- classification reports for both modalities
- training-history subplots
- retrieval metrics for:
  - SAR → SAR
  - Optical → Optical
  - SAR → Optical
  - Optical → SAR
- Precision@K, Recall@K and F1@K
- average retrieval time per query
- random retrieval examples for all four directions
- inference-ready files
- complete result-directory upload to Kaggle


In [ ]:
!pip install -q -U kagglehub scikit-learn

## 1. Imports and configuration

In [ ]:
import json
import os
import random
import time
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from kaggle_secrets import UserSecretsClient
from sklearn.metrics import classification_report
from tqdm.auto import tqdm

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

tf.keras.mixed_precision.set_global_policy("float32")
tf.config.optimizer.set_jit(False)

TFRECORD_HANDLE = "glitchr/eurosat-rgb-ms-sar-paired-tfrecords"

OUT_DIR = Path("/kaggle/working/eurodata_sar_optical_hybrid_results")
MODEL_DIR = OUT_DIR / "models"
REPORT_DIR = OUT_DIR / "reports"
PLOT_DIR = OUT_DIR / "plots"
EMB_DIR = OUT_DIR / "embeddings"

for directory in [OUT_DIR, MODEL_DIR, REPORT_DIR, PLOT_DIR, EMB_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 64
RGB_CHANNELS = 3
SAR_CHANNELS = 2
NUM_CLASSES = 10
BATCH_SIZE = 64
EPOCHS = 100
EMBED_DIM = 256
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
TRIPLET_MARGIN = 0.2
TEMPERATURE = 1.0

K_LIST = [1, 5, 10, 20, 50]

CLASS_NAMES = [
    "AnnualCrop",
    "Forest",
    "HerbaceousVegetation",
    "Highway",
    "Industrial",
    "Pasture",
    "PermanentCrop",
    "Residential",
    "River",
    "SeaLake",
]

print("TensorFlow:", tf.__version__)
print("Output directory:", OUT_DIR)

## 2. Download and load paired TFRecords

In [ ]:
downloaded_dir = Path(kagglehub.dataset_download(TFRECORD_HANDLE))

candidate_roots = [downloaded_dir]
candidate_roots.extend(
    path.parent
    for path in downloaded_dir.rglob("manifest.json")
)

TFREC_DIR = next(
    root
    for root in candidate_roots
    if (root / "manifest.json").exists()
)

manifest = json.loads(
    (TFREC_DIR / "manifest.json").read_text()
)

SAR_MEAN = np.asarray(
    manifest["sar_normalization"]["mean"],
    dtype=np.float32,
)

SAR_STD = np.asarray(
    manifest["sar_normalization"]["std"],
    dtype=np.float32,
)

SAR_MEAN_T = tf.constant(
    SAR_MEAN.reshape(1, 1, -1),
    dtype=tf.float32,
)

SAR_STD_T = tf.constant(
    SAR_STD.reshape(1, 1, -1),
    dtype=tf.float32,
)

train_df = pd.read_csv(TFREC_DIR / "train_metadata.csv")
validation_df = pd.read_csv(TFREC_DIR / "validation_metadata.csv")
test_df = pd.read_csv(TFREC_DIR / "test_metadata.csv")

print("TFRecord directory:", TFREC_DIR)
print("Train examples:", len(train_df))
print("Validation examples:", len(validation_df))
print("Test examples:", len(test_df))

In [ ]:
FEATURE_SPEC = {
    "rgb_raw": tf.io.FixedLenFeature([], tf.string),
    "ms_raw": tf.io.FixedLenFeature([], tf.string),
    "sar_raw": tf.io.FixedLenFeature([], tf.string),
    "target": tf.io.FixedLenFeature([], tf.int64),
    "file_id": tf.io.FixedLenFeature([], tf.string),
    "label": tf.io.FixedLenFeature([], tf.string),
}


def resolve_shard_paths(split):
    shard_names = manifest["splits"][split]["shards"]
    paths = []

    for shard_name in shard_names:
        shard_path = TFREC_DIR / shard_name

        if not shard_path.exists() and shard_name.endswith(".gz"):
            shard_path = TFREC_DIR / shard_name.removesuffix(".gz")

        if not shard_path.exists() and shard_name.endswith(".tfrecord"):
            shard_path = TFREC_DIR / f"{shard_name}.gz"

        if not shard_path.exists():
            raise FileNotFoundError(shard_name)

        paths.append(str(shard_path))

    compression_type = "GZIP" if all(path.endswith(".gz") for path in paths) else ""
    return paths, compression_type


def decode_record(serialized):
    item = tf.io.parse_single_example(serialized, FEATURE_SPEC)

    rgb = tf.io.parse_tensor(item["rgb_raw"], out_type=tf.uint8)
    sar = tf.io.parse_tensor(item["sar_raw"], out_type=tf.float16)

    rgb = tf.ensure_shape(rgb, [IMAGE_SIZE, IMAGE_SIZE, RGB_CHANNELS])
    sar = tf.ensure_shape(sar, [IMAGE_SIZE, IMAGE_SIZE, SAR_CHANNELS])

    rgb = tf.cast(rgb, tf.float32) / 255.0

    sar = tf.cast(sar, tf.float32)
    sar = (sar - SAR_MEAN_T) / tf.maximum(SAR_STD_T, 1e-3)
    sar = tf.clip_by_value(sar, -5.0, 5.0)

    tf.debugging.assert_all_finite(rgb, "RGB tensor contains NaN or infinity.")
    tf.debugging.assert_all_finite(sar, "SAR tensor contains NaN or infinity.")

    target = tf.cast(item["target"], tf.int32)

    return {
        "rgb": rgb,
        "sar": sar,
        "file_id": item["file_id"],
        "label": item["label"],
    }, target


In [ ]:
def paired_augment(inputs, target):
    rgb = inputs["rgb"]
    sar = inputs["sar"]

    seed = tf.random.uniform([2], maxval=2**31 - 1, dtype=tf.int32)

    rgb = tf.image.stateless_random_flip_left_right(rgb, seed)
    sar = tf.image.stateless_random_flip_left_right(sar, seed)

    second_seed = seed + tf.constant([17, 29], dtype=tf.int32)
    rgb = tf.image.stateless_random_flip_up_down(rgb, second_seed)
    sar = tf.image.stateless_random_flip_up_down(sar, second_seed)

    rotation_k = tf.random.stateless_uniform(
        [],
        seed + tf.constant([31, 43], dtype=tf.int32),
        minval=0,
        maxval=4,
        dtype=tf.int32,
    )

    rgb = tf.image.rot90(rgb, rotation_k)
    sar = tf.image.rot90(sar, rotation_k)

    rgb = tf.image.stateless_random_brightness(rgb, max_delta=0.05, seed=seed + 59)
    rgb = tf.image.stateless_random_contrast(
        rgb,
        lower=0.9,
        upper=1.1,
        seed=seed + 71,
    )
    rgb = tf.clip_by_value(rgb, 0.0, 1.0)

    sar_scale = tf.random.stateless_uniform(
        [1, 1, SAR_CHANNELS],
        seed=seed + 83,
        minval=0.97,
        maxval=1.03,
    )
    sar = tf.clip_by_value(sar * sar_scale, -5.0, 5.0)

    return {
        "rgb": rgb,
        "sar": sar,
        "file_id": inputs["file_id"],
        "label": inputs["label"],
    }, target


def make_dataset(split, training=False, include_metadata=False):
    shard_paths, compression_type = resolve_shard_paths(split)

    dataset = tf.data.TFRecordDataset(
        shard_paths,
        compression_type=compression_type,
        num_parallel_reads=tf.data.AUTOTUNE,
    )

    dataset = dataset.map(
        decode_record,
        num_parallel_calls=tf.data.AUTOTUNE,
        deterministic=not training,
    )

    if training:
        dataset = dataset.shuffle(
            min(int(manifest["splits"][split]["examples"]), 8192),
            seed=SEED,
            reshuffle_each_iteration=True,
        )
        dataset = dataset.map(
            paired_augment,
            num_parallel_calls=tf.data.AUTOTUNE,
            deterministic=False,
        )

    if not include_metadata:
        dataset = dataset.map(
            lambda inputs, target: (
                {"rgb": inputs["rgb"], "sar": inputs["sar"]},
                target,
            ),
            num_parallel_calls=tf.data.AUTOTUNE,
            deterministic=not training,
        )

    dataset = dataset.batch(BATCH_SIZE, drop_remainder=training)
    return dataset.prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset("train", training=True)
validation_ds = make_dataset("validation")
test_ds = make_dataset("test")
test_metadata_ds = make_dataset("test", include_metadata=True)

sample_inputs, sample_targets = next(iter(train_ds))

print("RGB batch:", sample_inputs["rgb"].shape)
print("SAR batch:", sample_inputs["sar"].shape)
print("Targets:", sample_targets.shape)


## 3. Hybrid encoder

In [ ]:
layers = tf.keras.layers


@tf.keras.utils.register_keras_serializable(package="EuroData")
class SqueezeExcitation(layers.Layer):
    def __init__(self, reduction=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        channels = int(input_shape[-1])
        reduced_channels = max(channels // self.reduction, 8)

        self.reduce_conv = layers.Conv2D(
            reduced_channels,
            kernel_size=1,
            activation="relu",
        )

        self.expand_conv = layers.Conv2D(
            channels,
            kernel_size=1,
            activation="sigmoid",
        )

        super().build(input_shape)

    def call(self, inputs):
        weights = tf.reduce_mean(
            inputs,
            axis=[1, 2],
            keepdims=True,
        )

        weights = self.reduce_conv(weights)
        weights = self.expand_conv(weights)

        return inputs * weights


@tf.keras.utils.register_keras_serializable(package="EuroData")
class CoordinateAttention(layers.Layer):
    def __init__(self, reduction=8, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        channels = int(input_shape[-1])
        reduced_channels = max(channels // self.reduction, 8)

        self.shared_conv = layers.Conv2D(
            reduced_channels,
            kernel_size=1,
            use_bias=False,
        )

        self.shared_bn = layers.BatchNormalization()

        self.height_conv = layers.Conv2D(
            channels,
            kernel_size=1,
            activation="sigmoid",
        )

        self.width_conv = layers.Conv2D(
            channels,
            kernel_size=1,
            activation="sigmoid",
        )

        super().build(input_shape)

    def call(self, inputs, training=None):
        shape = tf.shape(inputs)
        height = shape[1]
        width = shape[2]

        height_context = tf.reduce_mean(
            inputs,
            axis=2,
            keepdims=True,
        )

        width_context = tf.transpose(
            tf.reduce_mean(
                inputs,
                axis=1,
                keepdims=True,
            ),
            [0, 2, 1, 3],
        )

        combined = tf.concat(
            [height_context, width_context],
            axis=1,
        )

        combined = self.shared_conv(combined)
        combined = self.shared_bn(combined, training=training)
        combined = tf.nn.silu(combined)

        height_features = combined[:, :height]
        width_features = tf.transpose(
            combined[:, height:height + width],
            [0, 2, 1, 3],
        )

        height_attention = self.height_conv(height_features)
        width_attention = self.width_conv(width_features)

        return inputs * height_attention * width_attention


@tf.keras.utils.register_keras_serializable(package="EuroData")
class BalancedAttention(layers.Layer):
    def build(self, input_shape):
        self.coordinate_attention = CoordinateAttention()
        self.squeeze_excitation = SqueezeExcitation()

        self.fusion_logit = self.add_weight(
            name="fusion_logit",
            shape=(),
            initializer="zeros",
            trainable=True,
        )

        super().build(input_shape)

    def call(self, inputs, training=None):
        spatial_features = self.coordinate_attention(
            inputs,
            training=training,
        )

        spectral_features = self.squeeze_excitation(inputs)

        spatial_weight = tf.sigmoid(self.fusion_logit)

        return (
            spatial_weight * spatial_features
            + (1.0 - spatial_weight) * spectral_features
        )


@tf.keras.utils.register_keras_serializable(package="EuroData")
class AttentionPooling(layers.Layer):
    def __init__(self, output_dim=256, num_heads=8, **kwargs):
        super().__init__(**kwargs)
        self.output_dim = output_dim
        self.num_heads = num_heads

    def build(self, input_shape):
        self.input_projection = layers.Dense(self.output_dim)

        self.attention = layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=self.output_dim // self.num_heads,
            output_shape=self.output_dim,
        )

        self.output_projection = layers.Dense(self.output_dim)

        self.query_token = self.add_weight(
            name="query_token",
            shape=(1, 1, self.output_dim),
            initializer="glorot_uniform",
            trainable=True,
        )

        super().build(input_shape)

    def call(self, inputs, training=None):
        inputs = tf.cast(inputs, tf.float32)
        shape = tf.shape(inputs)

        tokens = tf.reshape(
            inputs,
            [shape[0], shape[1] * shape[2], shape[3]],
        )

        tokens = self.input_projection(tokens)

        query = tf.tile(
            self.query_token,
            [shape[0], 1, 1],
        )

        pooled = self.attention(
            query=query,
            key=tokens,
            value=tokens,
            training=training,
        )

        pooled = tf.squeeze(pooled, axis=1)

        return self.output_projection(pooled)

In [ ]:
EFFICIENTNET_VARIANT = "B0"  # Recommended for 64×64 images and low retrieval latency.
EFFICIENTNET_BUILDERS = {
    "B0": tf.keras.applications.EfficientNetV2B0,
    "B1": tf.keras.applications.EfficientNetV2B1,
    "B2": tf.keras.applications.EfficientNetV2B2,
    "B3": tf.keras.applications.EfficientNetV2B3,
}


def build_encoder(input_shape, name):
    if EFFICIENTNET_VARIANT not in EFFICIENTNET_BUILDERS:
        raise ValueError(
            f"Unsupported EfficientNet variant: {EFFICIENTNET_VARIANT}. "
            f"Choose from {list(EFFICIENTNET_BUILDERS)}."
        )

    inputs = tf.keras.Input(shape=input_shape, name=f"{name}_input")
    backbone_builder = EFFICIENTNET_BUILDERS[EFFICIENTNET_VARIANT]

    # weights=None supports both 3-channel optical and 2-channel SAR inputs.
    # The input pipeline already normalizes each modality.
    backbone = backbone_builder(
        include_top=False,
        weights=None,
        input_shape=input_shape,
        include_preprocessing=False,
        name=f"{name}_efficientnetv2_{EFFICIENTNET_VARIANT.lower()}",
    )

    x = backbone(inputs)
    x = BalancedAttention(name=f"{name}_balanced_attention")(x)
    x = AttentionPooling(
        output_dim=EMBED_DIM,
        num_heads=8,
        name=f"{name}_attention_pool",
    )(x)
    x = layers.Dropout(0.20, name=f"{name}_embedding_dropout")(x)
    x = layers.Dense(EMBED_DIM, use_bias=False, name=f"{name}_embedding_dense")(x)
    x = layers.BatchNormalization(name=f"{name}_embedding_bn")(x)
    embeddings = layers.UnitNormalization(axis=-1, name=f"{name}_embedding")(x)

    return tf.keras.Model(inputs, embeddings, name=name)


optical_encoder = build_encoder(
    (IMAGE_SIZE, IMAGE_SIZE, RGB_CHANNELS),
    "optical_encoder",
)

sar_encoder = build_encoder(
    (IMAGE_SIZE, IMAGE_SIZE, SAR_CHANNELS),
    "sar_encoder",
)

print("EfficientNet variant:", EFFICIENTNET_VARIANT)
print("Optical output:", optical_encoder.output_shape)
print("SAR output:", sar_encoder.output_shape)
print("Optical parameters:", f"{optical_encoder.count_params():,}")
print("SAR parameters:", f"{sar_encoder.count_params():,}")


## 4. Hybrid training model

In [ ]:
def batch_hard_triplet_loss(labels, embeddings, margin=TRIPLET_MARGIN):
    labels = tf.reshape(tf.cast(labels, tf.int32), [-1])
    embeddings = tf.cast(embeddings, tf.float32)

    squared_norm = tf.reduce_sum(tf.square(embeddings), axis=1)
    distances = tf.expand_dims(squared_norm, 1) - 2.0 * tf.matmul(embeddings, embeddings, transpose_b=True) + tf.expand_dims(squared_norm, 0)
    distances = tf.maximum(distances, 0.0)

    same_class = tf.equal(labels[:, None], labels[None, :])
    identity = tf.eye(tf.shape(labels)[0], dtype=tf.bool)

    positive_mask = tf.logical_and(same_class, tf.logical_not(identity))
    negative_mask = tf.logical_not(same_class)

    hardest_positive = tf.reduce_max(tf.where(positive_mask, distances, tf.zeros_like(distances)), axis=1)
    hardest_negative = tf.reduce_min(tf.where(negative_mask, distances, tf.fill(tf.shape(distances), tf.constant(1e9, tf.float32))), axis=1)

    valid_anchor = tf.logical_and(
        tf.reduce_any(positive_mask, axis=1),
        tf.reduce_any(negative_mask, axis=1),
    )

    losses = tf.maximum(hardest_positive - hardest_negative + margin, 0.0)
    valid_losses = tf.boolean_mask(losses, valid_anchor)

    return tf.cond(
        tf.size(valid_losses) > 0,
        lambda: tf.reduce_mean(valid_losses),
        lambda: tf.constant(0.0, tf.float32),
    )


def semantic_alignment_loss(optical_embeddings, sar_embeddings, labels, temperature=TEMPERATURE):
    labels = tf.reshape(tf.cast(labels, tf.int32), [-1])

    optical_embeddings = tf.math.l2_normalize(optical_embeddings, axis=-1)
    sar_embeddings = tf.math.l2_normalize(sar_embeddings, axis=-1)

    logits = tf.matmul(optical_embeddings, sar_embeddings, transpose_b=True) / temperature

    positive_mask = tf.cast(tf.equal(labels[:, None], labels[None, :]), tf.float32)
    targets = positive_mask / tf.maximum(tf.reduce_sum(positive_mask, axis=1, keepdims=True), 1.0)

    optical_to_sar = tf.reduce_mean(
        tf.nn.softmax_cross_entropy_with_logits(labels=targets, logits=logits)
    )

    sar_to_optical = tf.reduce_mean(
        tf.nn.softmax_cross_entropy_with_logits(labels=tf.transpose(targets), logits=tf.transpose(logits))
    )

    return 0.5 * (optical_to_sar + sar_to_optical)

In [ ]:
class HybridRetrievalModel(tf.keras.Model):
    def __init__(self):
        super().__init__()

        self.optical_encoder = optical_encoder
        self.sar_encoder = sar_encoder

        self.optical_classifier = layers.Dense(NUM_CLASSES, name="optical_classifier")
        self.sar_classifier = layers.Dense(NUM_CLASSES, name="sar_classifier")

        self.loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.triplet_tracker = tf.keras.metrics.Mean(name="triplet_loss")
        self.classification_tracker = tf.keras.metrics.Mean(name="classification_loss")
        self.alignment_tracker = tf.keras.metrics.Mean(name="alignment_loss")

        self.optical_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name="optical_accuracy")
        self.sar_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name="sar_accuracy")

    @property
    def metrics(self):
        return [
            self.loss_tracker,
            self.triplet_tracker,
            self.classification_tracker,
            self.alignment_tracker,
            self.optical_accuracy,
            self.sar_accuracy,
        ]

    def call(self, inputs, training=False):
        optical_embeddings = self.optical_encoder(inputs["rgb"], training=training)
        sar_embeddings = self.sar_encoder(inputs["sar"], training=training)

        return {
            "optical_embedding": optical_embeddings,
            "sar_embedding": sar_embeddings,
            "optical_logits": self.optical_classifier(optical_embeddings),
            "sar_logits": self.sar_classifier(sar_embeddings),
        }

    def compute_losses(self, inputs, labels, training):
        outputs = self(inputs, training=training)

        optical_triplet = batch_hard_triplet_loss(labels, outputs["optical_embedding"])
        sar_triplet = batch_hard_triplet_loss(labels, outputs["sar_embedding"])

        triplet = 0.5 * (optical_triplet + sar_triplet)

        optical_ce = tf.reduce_mean(
            tf.keras.losses.sparse_categorical_crossentropy(
                labels,
                outputs["optical_logits"],
                from_logits=True,
            )
        )

        sar_ce = tf.reduce_mean(
            tf.keras.losses.sparse_categorical_crossentropy(
                labels,
                outputs["sar_logits"],
                from_logits=True,
            )
        )

        classification = 0.5 * (optical_ce + sar_ce)

        alignment = semantic_alignment_loss(
            outputs["optical_embedding"],
            outputs["sar_embedding"],
            labels,
        )

        total = triplet + classification + alignment

        if self.losses:
            total += tf.add_n(self.losses)

        return total, triplet, classification, alignment, outputs

    def update_trackers(self, total, triplet, classification, alignment, labels, outputs):
        self.loss_tracker.update_state(total)
        self.triplet_tracker.update_state(triplet)
        self.classification_tracker.update_state(classification)
        self.alignment_tracker.update_state(alignment)

        self.optical_accuracy.update_state(labels, outputs["optical_logits"])
        self.sar_accuracy.update_state(labels, outputs["sar_logits"])

    def train_step(self, data):
        inputs, labels = data

        with tf.GradientTape() as tape:
            values = self.compute_losses(inputs, labels, training=True)

        gradients = tape.gradient(values[0], self.trainable_variables)

        gradient_pairs = [
            (gradient, variable)
            for gradient, variable in zip(gradients, self.trainable_variables)
            if gradient is not None
        ]

        self.optimizer.apply_gradients(gradient_pairs)

        self.update_trackers(*values[:4], labels, values[4])

        return {metric.name: metric.result() for metric in self.metrics}

    def test_step(self, data):
        inputs, labels = data

        values = self.compute_losses(inputs, labels, training=False)

        self.update_trackers(*values[:4], labels, values[4])

        return {metric.name: metric.result() for metric in self.metrics}


model = HybridRetrievalModel()

_ = model(
    {
        "rgb": sample_inputs["rgb"][:2],
        "sar": sample_inputs["sar"][:2],
    },
    training=False,
)

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    global_clipnorm=1.0,
)

model.compile(
    optimizer=optimizer,
    jit_compile=False,
)

CHECKPOINT_PATH = MODEL_DIR / "best.weights.h5"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        CHECKPOINT_PATH,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(REPORT_DIR / "history.csv"),
    tf.keras.callbacks.TerminateOnNaN(),
]

history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

model.load_weights(CHECKPOINT_PATH)

optical_encoder.save(MODEL_DIR / "optical_encoder.keras")
sar_encoder.save(MODEL_DIR / "sar_encoder.keras")

## 5. Training history subplots

In [ ]:
history_df = pd.DataFrame(history.history)

history_metrics = [
    "loss",
    "triplet_loss",
    "classification_loss",
    "alignment_loss",
    "optical_accuracy",
    "sar_accuracy",
]

figure, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for axis, metric in zip(axes, history_metrics):
    if metric in history_df:
        axis.plot(history_df[metric], label=metric)

    validation_metric = f"val_{metric}"

    if validation_metric in history_df:
        axis.plot(history_df[validation_metric], label=validation_metric)

    axis.set_title(metric)
    axis.set_xlabel("Epoch")
    axis.grid(True, alpha=0.3)
    axis.legend()

plt.tight_layout()

plt.savefig(
    PLOT_DIR / "training_history_subplots.png",
    dpi=180,
)

plt.show()

## 6. Classification reports and embedding extraction

In [ ]:
def predict_test(dataset):
    targets = []
    optical_predictions = []
    sar_predictions = []
    optical_embeddings = []
    sar_embeddings = []
    file_ids = []
    labels = []

    for inputs, batch_targets in tqdm(dataset, desc="Running test inference"):
        outputs = model(
            {
                "rgb": inputs["rgb"],
                "sar": inputs["sar"],
            },
            training=False,
        )

        targets.append(batch_targets.numpy())
        optical_predictions.append(np.argmax(outputs["optical_logits"].numpy(), axis=1))
        sar_predictions.append(np.argmax(outputs["sar_logits"].numpy(), axis=1))
        optical_embeddings.append(outputs["optical_embedding"].numpy())
        sar_embeddings.append(outputs["sar_embedding"].numpy())
        file_ids.append(inputs["file_id"].numpy())

        labels.extend(
            value.decode("utf-8")
            for value in inputs["label"].numpy()
        )

    return {
        "y": np.concatenate(targets),
        "optical_pred": np.concatenate(optical_predictions),
        "sar_pred": np.concatenate(sar_predictions),
        "optical_emb": np.concatenate(optical_embeddings),
        "sar_emb": np.concatenate(sar_embeddings),
        "file_id": np.concatenate(file_ids),
        "label": np.asarray(labels),
    }


pred = predict_test(test_metadata_ds)

optical_report = pd.DataFrame(
    classification_report(
        pred["y"],
        pred["optical_pred"],
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
).T.reset_index(names="class")

sar_report = pd.DataFrame(
    classification_report(
        pred["y"],
        pred["sar_pred"],
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
).T.reset_index(names="class")

optical_report["modality"] = "Optical"
sar_report["modality"] = "SAR"

classification_report_df = pd.concat(
    [optical_report, sar_report],
    ignore_index=True,
)

classification_report_df.to_csv(
    REPORT_DIR / "classification_reports_sar_optical.csv",
    index=False,
)

display(classification_report_df)

## 7. Retrieval metrics for all four directions

In [ ]:
def retrieval_metrics(
    query_embeddings,
    gallery_embeddings,
    query_labels,
    gallery_labels,
    direction,
    k_list=K_LIST,
    same_set=False,
    timing_repeats=3,
):
    query_embeddings = query_embeddings / np.maximum(
        np.linalg.norm(query_embeddings, axis=1, keepdims=True), 1e-8
    )
    gallery_embeddings = gallery_embeddings / np.maximum(
        np.linalg.norm(gallery_embeddings, axis=1, keepdims=True), 1e-8
    )

    max_k = min(max(k_list), len(gallery_embeddings) - (1 if same_set else 0))

    # Warm-up prevents one-time BLAS initialization from inflating the result.
    _ = query_embeddings[0] @ gallery_embeddings.T

    elapsed_runs = []
    ranking = None
    scores = None

    for _ in range(timing_repeats):
        start_time = time.perf_counter()
        scores = query_embeddings @ gallery_embeddings.T

        if same_set:
            np.fill_diagonal(scores, -np.inf)

        candidate_indices = np.argpartition(
            -scores,
            kth=max_k - 1,
            axis=1,
        )[:, :max_k]
        candidate_scores = np.take_along_axis(scores, candidate_indices, axis=1)
        local_order = np.argsort(-candidate_scores, axis=1)
        current_ranking = np.take_along_axis(candidate_indices, local_order, axis=1)
        elapsed_runs.append(time.perf_counter() - start_time)
        ranking = current_ranking

    average_retrieval_time_seconds = float(
        np.median(elapsed_runs) / len(query_embeddings)
    )
    average_retrieval_time_ms = average_retrieval_time_seconds * 1000.0

    rows = []
    for k in k_list:
        if k > max_k:
            continue

        precision_scores, recall_scores, f1_scores = [], [], []

        for query_index in range(len(query_embeddings)):
            retrieved_indices = ranking[query_index, :k]
            relevant = gallery_labels[retrieved_indices] == query_labels[query_index]

            correct = int(relevant.sum())
            total_relevant = int(
                np.sum(gallery_labels == query_labels[query_index])
            ) - (1 if same_set else 0)

            precision = correct / k
            recall = correct / max(total_relevant, 1)
            f1 = 2.0 * precision * recall / max(precision + recall, 1e-12)

            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)

        rows.append({
            "direction": direction,
            "retrieval_type": "same-modal" if same_set else "cross-modal",
            "k": k,
            "precision": np.mean(precision_scores),
            "recall": np.mean(recall_scores),
            "f1": np.mean(f1_scores),
            "average_retrieval_time_seconds": average_retrieval_time_seconds,
            "average_retrieval_time_ms": average_retrieval_time_ms,
            "gallery_size": len(gallery_embeddings),
        })

    return pd.DataFrame(rows), scores, ranking


tasks = {
    "Optical_to_Optical": (pred["optical_emb"], pred["optical_emb"], True),
    "SAR_to_SAR": (
        pred["sar_emb"], pred["sar_emb"], True
    ),
    "Optical_to_SAR": (
        pred["optical_emb"], pred["sar_emb"], False
    ),
    "SAR_to_Optical": (
        pred["sar_emb"], pred["optical_emb"], False
    ),
}

metric_frames = []
retrieval_cache = {}

for direction, (query_embeddings, gallery_embeddings, same_set) in tasks.items():
    metrics_df, scores, ranking = retrieval_metrics(
        query_embeddings,
        gallery_embeddings,
        pred["label"],
        pred["label"],
        direction,
        K_LIST,
        same_set,
    )
    metric_frames.append(metrics_df)
    retrieval_cache[direction] = (scores, ranking)

retrieval_metrics_df = pd.concat(metric_frames, ignore_index=True)
retrieval_metrics_df.to_csv(
    REPORT_DIR / "retrieval_metrics_at_k.csv",
    index=False,
)

competition_metrics_df = retrieval_metrics_df[
    retrieval_metrics_df["k"].isin([5, 10])
][[
    "direction",
    "retrieval_type",
    "k",
    "f1",
    "average_retrieval_time_ms",
    "gallery_size",
]].copy()
competition_metrics_df.to_csv(
    REPORT_DIR / "competition_metrics_f1_and_time.csv",
    index=False,
)

retrieval_time_df = retrieval_metrics_df[[
    "direction",
    "retrieval_type",
    "average_retrieval_time_seconds",
    "average_retrieval_time_ms",
    "gallery_size",
]].drop_duplicates()
retrieval_time_df.to_csv(
    REPORT_DIR / "average_retrieval_time_per_query.csv",
    index=False,
)

display(competition_metrics_df)
display(retrieval_time_df)
display(
    retrieval_metrics_df.pivot(
        index="direction",
        columns="k",
        values=["precision", "recall", "f1"],
    )
)


## 8. Random test retrieval plots

In [ ]:
def percentile_stretch(channel):
    finite = channel[np.isfinite(channel)]

    if finite.size == 0:
        return np.zeros_like(channel, dtype=np.float32)

    minimum, maximum = np.percentile(finite, [2, 98])
    normalized = (np.nan_to_num(channel, nan=minimum) - minimum) / max(maximum - minimum, 1e-8)
    return np.clip(normalized, 0.0, 1.0)


def sar_preview(sar):
    vv = percentile_stretch(sar[..., 0])
    vh = percentile_stretch(sar[..., 1])
    return 0.5 * (vv + vh)


def collect_test_images(indices):
    required_indices = {int(i) for i in indices}
    collected = {}
    offset = 0

    for inputs, targets in test_metadata_ds:
        for batch_index in range(len(targets)):
            global_index = offset + batch_index

            if global_index in required_indices:
                collected[global_index] = {
                    "rgb": inputs["rgb"][batch_index].numpy(),
                    "sar": inputs["sar"][batch_index].numpy(),
                    "label": inputs["label"][batch_index].numpy().decode("utf-8"),
                }

        offset += len(targets)

        if len(collected) == len(required_indices):
            break

    missing = required_indices - collected.keys()
    if missing:
        raise ValueError(f"Missing test images: {sorted(missing)}")

    return collected


In [ ]:
TASK_INFO = {
    "SAR_to_SAR": {
        "title": "SAR → SAR Retrieval",
        "query_mode": "sar",
        "gallery_mode": "sar",
        "query_name": "SAR",
        "gallery_name": "SAR",
    },
    "Optical_to_Optical": {
        "title": "Optical → Optical Retrieval",
        "query_mode": "optical",
        "gallery_mode": "optical",
        "query_name": "Optical",
        "gallery_name": "Optical",
    },
    "SAR_to_Optical": {
        "title": "SAR → Optical Cross-Modal Retrieval",
        "query_mode": "sar",
        "gallery_mode": "optical",
        "query_name": "SAR",
        "gallery_name": "Optical",
    },
    "Optical_to_SAR": {
        "title": "Optical → SAR Cross-Modal Retrieval",
        "query_mode": "optical",
        "gallery_mode": "sar",
        "query_name": "Optical",
        "gallery_name": "SAR",
    },
}


In [ ]:
def display_retrieval_image(axis, sample, mode):
    if mode == "sar": axis.imshow(sar_preview(sample["sar"]), cmap="gray", vmin=0, vmax=1)
    else: axis.imshow(sample["rgb"])
    axis.set_xticks([]); axis.set_yticks([])


def set_retrieval_border(axis, color, linewidth=4):
    axis.set_frame_on(True)
    for spine in axis.spines.values(): spine.set_visible(True); spine.set_color(color); spine.set_linewidth(linewidth)


def plot_random_retrieval(direction, top_k=10, query_index=None):
    if direction not in TASK_INFO: raise ValueError(f"Unsupported retrieval direction: {direction}")
    if top_k > 10: raise ValueError("top_k must be 10 or lower for the three-row layout.")

    task = TASK_INFO[direction]
    scores, ranking = retrieval_cache[direction]
    if query_index is None: query_index = np.random.randint(len(pred["y"]))

    gallery_indices = ranking[query_index, :top_k]
    images = collect_test_images([query_index, *gallery_indices.tolist()])
    query = images[query_index]

    columns, total_rows = 5, 3
    figure, axes = plt.subplots(total_rows, columns, figsize=(15, 10))
    for axis in axes.flat: axis.axis("off")

    figure.suptitle(task["title"], fontsize=18, fontweight="bold", y=0.98)

    query_axis = axes[0, columns // 2]
    display_retrieval_image(query_axis, query, task["query_mode"])
    query_axis.axis("on"); query_axis.set_xticks([]); query_axis.set_yticks([])
    set_retrieval_border(query_axis, "blue", 5)
    query_axis.set_title(f"Query: {task['query_name']}\nClass: {query['label']}", fontsize=12, fontweight="bold", color="blue")

    rows = []
    for rank_position, gallery_index in enumerate(gallery_indices, start=1):
        plot_row, plot_column = (rank_position - 1) // columns + 1, (rank_position - 1) % columns
        axis = axes[plot_row, plot_column]
        gallery_index = int(gallery_index)
        gallery = images[gallery_index]

        display_retrieval_image(axis, gallery, task["gallery_mode"])
        axis.axis("on"); axis.set_xticks([]); axis.set_yticks([])

        relevant = int(gallery["label"] == query["label"])
        border_color = "green" if relevant else "red"
        result_text = "Correct" if relevant else "Wrong"

        set_retrieval_border(axis, border_color, 4)
        score = float(scores[query_index, gallery_index])

        axis.set_title(
            f"Rank {rank_position}: {task['gallery_name']}\n{gallery['label']} | {result_text}\nScore: {score:.3f}",
            fontsize=9, color=border_color, fontweight="bold"
        )

        rows.append({
            "direction": direction,
            "task": task["title"],
            "query_index": query_index,
            "query_modality": task["query_name"],
            "gallery_modality": task["gallery_name"],
            "rank": rank_position,
            "gallery_index": gallery_index,
            "query_label": query["label"],
            "gallery_label": gallery["label"],
            "score": score,
            "relevant": relevant,
            "result": result_text,
        })

    figure.text(0.5, 0.02, "Blue: Query     Green: Correct match     Red: Wrong match", ha="center", fontsize=11, fontweight="bold")
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    plt.savefig(PLOT_DIR / f"{direction}_query_{query_index}_top_{top_k}.png", dpi=180, bbox_inches="tight")
    plt.show()

    return pd.DataFrame(rows)

In [ ]:
sample_frames = []

for direction in [
    "SAR_to_SAR",
    "Optical_to_Optical",
    "SAR_to_Optical",
    "Optical_to_SAR",
]:
    result_df = plot_random_retrieval(
        direction=direction,
        top_k=10,
    )

    sample_frames.append(
        result_df
    )

sample_results = pd.concat(
    sample_frames,
    ignore_index=True,
)

sample_results.to_csv(
    REPORT_DIR
    / "random_retrieval_samples.csv",
    index=False,
)

display(
    sample_results
)

## 9. Save inference-ready artifacts

In [ ]:
np.save(EMB_DIR / "test_sar_embeddings.npy", pred["sar_emb"])
np.save(EMB_DIR / "test_optical_embeddings.npy", pred["optical_emb"])
np.save(EMB_DIR / "test_targets.npy", pred["y"])
np.save(EMB_DIR / "test_file_ids.npy", pred["file_id"])

test_df.to_csv(REPORT_DIR / "test_metadata.csv", index=False)

config = {
    "image_size": IMAGE_SIZE,
    "sar_channels": SAR_CHANNELS,
    "sar_band_order": ["VV", "VH"],
    "sar_mean": SAR_MEAN.tolist(),
    "sar_std": SAR_STD.tolist(),
    "rgb_channels": RGB_CHANNELS,
    "embedding_dim": EMBED_DIM,
    "backbone": f"EfficientNetV2-{EFFICIENTNET_VARIANT}",
    "class_names": CLASS_NAMES,
    "k_list": K_LIST,
}

(OUT_DIR / "inference_config.json").write_text(json.dumps(config, indent=2))

summary = {
    "model": f"EfficientNetV2-{EFFICIENTNET_VARIANT} Hybrid Attention",
    "tasks": list(tasks),
    "classification_report": "reports/classification_reports_sar_optical.csv",
    "retrieval_metrics": "reports/retrieval_metrics_at_k.csv",
    "competition_metrics": "reports/competition_metrics_f1_and_time.csv",
    "retrieval_time": "reports/average_retrieval_time_per_query.csv",
    "sar_encoder": "models/sar_encoder.keras",
    "optical_encoder": "models/optical_encoder.keras",
}

(OUT_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2))

print("Saved inference artifacts to", OUT_DIR)


## 10. Upload result directory to Kaggle

In [ ]:
user_secrets = UserSecretsClient()

os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = user_secrets.get_secret("KAGGLE_KEY")

kaggle_username = os.environ["KAGGLE_USERNAME"].strip()

RESULT_HANDLE = (
    f"{kaggle_username}/"
    "eurosat-sar-optical-hybrid-retrieval-results"
)

kagglehub.dataset_upload(
    RESULT_HANDLE,
    OUT_DIR,
    ignore_patterns=["*.tmp", "*.log"],
)

print("Uploaded:", RESULT_HANDLE)